In [141]:
import os

import json
import glob
import re
import random
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from datetime import date

In [142]:
random.seed = 42

# BioRED Train & GT Loading & Parsing

### Load Key

In [2]:
load_dotenv()
print(os.getenv("OPEN_AI_TEST_KEY")[:15])

sk-proj-4WDSBIA


In [3]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_TEST_KEY")
)

### Load & Downsample BioRED Train Abstracts

In [4]:
biored_train = pd.read_csv("../data/processed/biored/br_train.csv")

biored_train_sample = biored_train.sample(
    n=35,
    random_state=42
).copy()

### BioRED GT Parsing

In [145]:
sampled_pmids = set(biored_train_sample["pmid"])

biored_train_gts = pd.read_csv("../data/processed/biored/br_train_entity_relations.csv")

biored_train_gts_filtered = biored_train_gts[biored_train_gts["pmid"].isin(sampled_pmids)]

### Parse Into Two Sets of Tuples
* `predictions_entities`
* `predictions_relationships`

**Includes PMID for duplicate handling as sets automatically remove duplicates**
* PMID avoids duplicates across papers

In [147]:
ground_truth_entities = (
    set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_1"].str.strip().str.lower(), biored_train_gts_filtered["entity_1_type"]))
    | set(zip(biored_train_gts_filtered["pmid"], biored_train_gts_filtered["entity_2"].str.strip().str.lower(), biored_train_gts_filtered["entity_2_type"]))
)

ground_truth_relationships = set(zip(
    biored_train_gts_filtered["pmid"],
    biored_train_gts_filtered["entity_1"].str.strip().str.lower(),
    biored_train_gts_filtered["relation"],
    biored_train_gts_filtered["entity_2"].str.strip().str.lower()
))

### Few-Shot Construction

In [179]:
# Exclude the 35 pmids you're evaluating on
non_eval_pmids = set(biored_train["pmid"]) - sampled_pmids
few_shot_pmids = random.sample(list(non_eval_pmids), 3)

def build_few_shot_example(pmid, example_num):
    abstract = biored_train[biored_train["pmid"] == pmid]["abstract"].iloc[0]
    gt_rows = biored_train_gts[biored_train_gts["pmid"] == pmid]

    entities = list({
        (row["entity_1"], row["entity_1_type"]) for _, row in gt_rows.iterrows()
    } | {
        (row["entity_2"], row["entity_2_type"]) for _, row in gt_rows.iterrows()
    })

    relationships = [
        {"source": row["entity_1"], "relation": row["relation"], "target": row["entity_2"]}
        for _, row in gt_rows.iterrows()
    ]

    output_json = {
        "entities": [{"text": e[0], "type": e[1]} for e in entities],
        "relationships": relationships
    }

    return f"## EXAMPLE {example_num+1}:\n\n### Abstract:\n\n{abstract}\n\n### Correct BioRED annotation:\n\n{json.dumps(output_json, indent=2)}"

few_shot_block = "\n\n".join(build_few_shot_example(pmid, i) for i, pmid in enumerate(few_shot_pmids))

print(few_shot_block[:1750])  # sanity check

## EXAMPLE 1:

### Abstract:

To determine the incidence of clinically significant adverse events after long-term, fixed-dose, generic highly active antiretroviral therapy (HAART) use among HIV-infected individuals in South India, we examined the experiences of 3154 HIV-infected individuals who received a minimum of 3 months of generic HAART between February 1996 and December 2006 at a tertiary HIV care referral center in South India. The most common regimens were 3TC + d4T + nevirapine (NVP) (54.8%), zidovudine (AZT) + 3TC + NVP (14.5%), 3TC + d4T + efavirenz (EFV) (20.1%), and AZT + 3TC + EFV (5.4%). The most common adverse events and median CD4 at time of event were rash (15.2%; CD4, 285 cells/microL) and peripheral neuropathy (9.0% and 348 cells/microL). Clinically significant anemia (hemoglobin <7 g/dL) was observed in 5.4% of patients (CD4, 165 cells/microL) and hepatitis (clinical jaundice with alanine aminotransferase > 5 times upper limits of normal) in 3.5% of patients (CD4, 

# LLM API Call
##### **IMPORTANT** : `run_notes`, `prompt_version`, `parser_version` and `dataset` will all have to be manually set for each run if refined

In [184]:
with open("../data/prompt_refinement/prompt_versions.json", "r") as f:
    PROMPTS = json.load(f)

In [185]:
outputs = []

for index, row in biored_train_sample.iterrows():
    abstract = row["abstract"]

    prompt = (
        PROMPTS["v3"]["template"]
        .replace("{abstract}", abstract)
        .replace("{few_shot_block}", few_shot_block)
    )

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    outputs.append({
        "pmid": row["pmid"],
        "output": response.output_text
    })

In [203]:
run_notes = "Added few-shot prompting on top of prompt v2."

In [ ]:
run_dictionary = {
    "outputs": outputs,
    "run_notes": "Added few-shot prompting on top of prompt v2.",
    "prompt_version": "v3"
}

### Parse `outputs` list to JSON

In [188]:
extractions = []
parse_failures = 0

for item in outputs:
    try:
        parsed = json.loads(item["output"])
    except json.JSONDecodeError:
        parse_failures += 1
        parsed = {"entities": [], "relationships": []}

    extractions.append({
        "pmid": item["pmid"],
        "entities": parsed.get("entities", []),
        "relationships": parsed.get("relationships", [])
    })

print(f"Parsed {len(extractions)} extractions, {parse_failures} failed to parse as JSON")

Parsed 35 extractions, 0 failed to parse as JSON


### Parse Prediction Extractions (JSON) to Sets of Tuples

In [189]:
predictions_entities = {
    (extraction["pmid"], e["text"].strip().lower(), e["type"])
    for extraction in extractions
    for e in extraction["entities"]
}

predictions_relationships = {
    (
        extraction["pmid"],
        r["source"].strip().lower(),
        r["relation"],
        r["target"].strip().lower()
    )
    for extraction in extractions
    for r in extraction["relationships"]
}

# Evaluation Metrics

In [190]:
def compute_prf(predictions, ground_truth):
    tp = len(predictions & ground_truth)
    fp = len(predictions - ground_truth)
    fn = len(ground_truth - predictions)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

In [191]:
def span_matches(predicted: str, ground_truth: str) -> bool:
    predicted = predicted.strip().lower()
    ground_truth = ground_truth.strip().lower()

    return (
        predicted == ground_truth
        or predicted in ground_truth
        or ground_truth in predicted
    )

def compute_prf_relaxed(predictions, ground_truth, match_fn):
    matched_gt = set()
    tp = 0

    for p in predictions:
        for g in ground_truth:
            if g in matched_gt:
                continue
            if match_fn(p, g):
                matched_gt.add(g)
                tp += 1
                break

    fp = len(predictions) - tp
    fn = len(ground_truth) - len(matched_gt)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

In [192]:
def entity_match(pred, gt):
    return pred[0] == gt[0] and pred[2] == gt[2] and span_matches(pred[1], gt[1])

def relationship_match(pred, gt):
    return (
        pred[0] == gt[0]
        and pred[2] == gt[2]
        and span_matches(pred[1], gt[1])
        and span_matches(pred[3], gt[3])
    )

In [193]:
def entity_match(pred, gt):
    return pred[0] == gt[0] and pred[2] == gt[2] and span_matches(pred[1], gt[1])

In [194]:
entity_metrics_strict = compute_prf(predictions_entities, ground_truth_entities)
entity_metrics_relaxed = compute_prf_relaxed(predictions_entities, ground_truth_entities, entity_match)

relationship_metrics_strict = compute_prf(predictions_relationships, ground_truth_relationships)
relationship_metrics_relaxed = compute_prf_relaxed(predictions_relationships, ground_truth_relationships, relationship_match)

print("Strict:")
print("Entities:", entity_metrics_strict)
print("Relationships:", relationship_metrics_strict)
print("")
print("Relaxed:")
print("Entities:", entity_metrics_relaxed)
print("Relationships:", relationship_metrics_relaxed)

Strict:
Entities: {'precision': 0.3957, 'recall': 0.7137, 'f1': 0.5091, 'tp': 182, 'fp': 278, 'fn': 73}
Relationships: {'precision': 0.1425, 'recall': 0.1959, 'f1': 0.165, 'tp': 58, 'fp': 349, 'fn': 238}

Relaxed:
Entities: {'precision': 0.4826, 'recall': 0.8706, 'f1': 0.621, 'tp': 222, 'fp': 238, 'fn': 33}
Relationships: {'precision': 0.2555, 'recall': 0.3514, 'f1': 0.2959, 'tp': 104, 'fp': 303, 'fn': 192}


# Export Run

### Create Run Storage Dir.

In [195]:
os.makedirs("../data/exploration", exist_ok=True)

### Get next `run_id`

In [196]:
def get_next_run_id(prompt_runs_dir="../prompt_runs"):
    os.makedirs(prompt_runs_dir, exist_ok=True)
    existing = glob.glob(os.path.join(prompt_runs_dir, "run_*"))
    nums = [int(re.search(r"run_(\d+)", d).group(1)) for d in existing if re.search(r"run_(\d+)", d)]
    next_num = max(nums, default=0) + 1
    return f"run_{next_num:03d}"

run_id = get_next_run_id()
run_id

'run_004'

### Build and save run log
* NB: `prompt_version` & `parser_version` will have to be manually updated per prompt or parser refinement step
    * `dataset` will also have to be manually updated to avoid over-engineered code

In [197]:
run_log = {
    "run_id": run_id,
    "date": str(date.today()),
    "config": {
        "dataset": "biored_train",
        "num_abstracts": len(extractions),
        "prompt_version": "v3",
        "parser_version": "v2"
    },
    "metrics": {
        "entity": {
            "strict": entity_metrics_strict,
            "relaxed": entity_metrics_relaxed
        },
        "relation": {
            "strict": relationship_metrics_strict,
            "relaxed": relationship_metrics_relaxed
        }
    },
    "notes": run_notes,
    "prompt": prompt,
    "extractions": {
        "entities": list(predictions_entities),
        "relations": list(predictions_relationships)
    }
}

run_dir = "../prompt_runs"
os.makedirs(run_dir, exist_ok=True)

with open(os.path.join(run_dir, f"{run_id}.json"), "w") as f:
    json.dump(run_log, f, indent=2, default=list)  # default=list handles sets/tuples

print(f"Saved {run_id} to {run_dir}")

Saved run_004 to ../prompt_runs


In [199]:
def sample_errors(predictions, ground_truth, n=20, label="items"):
    false_positives = list(predictions - ground_truth)
    false_negatives = list(ground_truth - predictions)

    fp_sample = random.sample(false_positives, min(n, len(false_positives)))
    fn_sample = random.sample(false_negatives, min(n, len(false_negatives)))

    print(f"--- {label}: False Positives (predicted, not in ground truth) ---")
    print(f"Sampled {len(fp_sample)} of {len(false_positives)} total FPs\n")
    for item in fp_sample:
        print(" ", item)

    print(f"\n--- {label}: False Negatives (in ground truth, not predicted) ---")
    print(f"Sampled {len(fn_sample)} of {len(false_negatives)} total FNs\n")
    for item in fn_sample:
        print(" ", item)

    return fp_sample, fn_sample

In [206]:
relation_fp_sample, relation_fn_sample = sample_errors(
    predictions_relationships,
    ground_truth_relationships,
    n=20,
    label="Relationships"
)

--- Relationships: False Positives (predicted, not in ground truth) ---
Sampled 20 of 349 total FPs

  (24743235, 'csf-1', 'Positive_Correlation', 'cd11c+ cell expansion')
  (24341598, 'iodinated contrast agent', 'Positive_Correlation', 'contrast-induced nephropathy')
  (17192049, 'w1/m1', 'Positive_Correlation', 'prostate cancer')
  (25305591, 'experimental autoimmune encephalomyelitis', 'Negative_Correlation', 'il-10')
  (25305591, 'pituitary adenylyl cyclase-activating polypeptide', 'Negative_Correlation', 'experimental autoimmune encephalomyelitis')
  (20510337, 'cisplatin', 'Positive_Correlation', 'tumor necrosis factor-alpha')
  (28428256, 'pga2', 'Negative_Correlation', 'inflammatory signaling')
  (18827003, 'glucocorticoid sensitivity', 'Association', 'cortisol excess')
  (15970799, 'slco1b1*15', 'Association', 'oatp1b1')
  (24477591, '-930g allele', 'Association', 'cigarette smoking')
  (16574712, 'mdma', 'Positive_Correlation', 'set shifting')
  (16574712, 'mdma', 'Positive_C

In [207]:
relation_fp_sample, relation_fn_sample = sample_errors(
    predictions_entities,
    ground_truth_entities,
    n=20,
    label="Entities"
)

--- Entities: False Positives (predicted, not in ground truth) ---
Sampled 20 of 278 total FPs

  (28428256, 'p120-catenin', 'GeneOrGeneProduct')
  (19108278, '(-)-propranolol', 'ChemicalEntity')
  (20510337, 'reduced glutathione', 'ChemicalEntity')
  (20431083, 'intracerebral hemorrhage', 'DiseaseOrPhenotypicFeature')
  (25305591, 'ifn-gamma', 'GeneOrGeneProduct')
  (28512644, 'group a (beta-hemolytic) streptococcus infection', 'DiseaseOrPhenotypicFeature')
  (24914936, 'skeletal abnormalities', 'DiseaseOrPhenotypicFeature')
  (17192049, 'early age of onset', 'DiseaseOrPhenotypicFeature')
  (28512644, 'cat c262', 'SequenceVariant')
  (15686794, 'sinus rhythm', 'DiseaseOrPhenotypicFeature')
  (24914936, 'grossly abnormal bone morphology', 'DiseaseOrPhenotypicFeature')
  (18827003, 'hgralpha', 'GeneOrGeneProduct')
  (10491763, 'insulin', 'ChemicalEntity')
  (19108278, 'free fatty acids', 'ChemicalEntity')
  (24477591, 'cytochrome b245 alpha', 'GeneOrGeneProduct')
  (21163864, 'left vent

In [210]:
from rapidfuzz import fuzz

fuzz.ratio("ischemia", "ischaemia")      # high
# fuzz.ratio("hypoperfusion", "ischemia")  # low

94.11764705882352

In [ ]:
31469710